# Run `amplitude_quest.py` (jax_constantB) on a Kaggle P100

**Before running anything**: Notebook settings (right sidebar) → Accelerator → **GPU P100**.

Do not use "GPU T4 x2": the Gauss-Newton/CG solve needs `float64` (GN tolerance 1e-10, CG tolerance 1e-26),
and T4 has roughly 1/32 the double-precision throughput of its single-precision rate — it will likely run
this workload *slower* than plain CPU. P100 is a compute-class card with a much better fp64:fp32 ratio
(~1:2) and is the right choice here.

Cell 3 checks this and warns if the wrong accelerator is selected.

In [ ]:
# MUST run before any process (this kernel, or a later `!python3 ...` subprocess) touches
# CUDA. JAX's default is to preallocate ~75% of TOTAL device memory the moment it's first
# used, regardless of what the computation actually needs -- and this notebook ends up using
# jax in *two* separate processes (the kernel itself, in the verification cell below, and the
# `!python3 amplitude_quest.py` subprocess further down), each of which would otherwise try to
# grab ~12GB of the P100's 16GB on its own. That's the source of any CUDA_ERROR_OUT_OF_MEMORY
# noise (with a repeated backoff-then-eventually-succeeds pattern): XLA retries at smaller and
# smaller sizes until one fits alongside whatever the other process already grabbed -- slow
# and not guaranteed to succeed. Disabling preallocation makes every process allocate on
# demand instead, which is all this problem actually needs (a few GB at most, even at the
# largest 96x96x192 grid).
%env XLA_PYTHON_CLIENT_PREALLOCATE=false
%env XLA_PYTHON_CLIENT_ALLOCATOR=platform

In [ ]:
import subprocess

name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
).decode().strip()
print("Detected GPU:", name)

if "P100" not in name:
    print(
        "\n*** WARNING: this session is not a P100 ('%s' detected). ***\n"
        "*** amplitude_quest.py needs float64; T4's fp64 throughput is ~1/32 of fp32. ***\n"
        "*** Go to Settings > Accelerator > GPU P100 and restart the session. ***" % name
    )

In [ ]:
# jax[cuda12] pulls the CUDA-enabled jaxlib PJRT plugin from PyPI; it only needs a
# recent-enough NVIDIA driver (already present on Kaggle's P100 image), not a matching
# system CUDA toolkit install.
#
# NOTE: pip will likely print a wall of red dependency-conflict text here (protobuf, numpy,
# etc. pinned differently by Kaggle's preinstalled TensorFlow/PyTorch). Those are warnings,
# not failures -- the install still succeeds. What matters is the next cell's check passing.
!pip install -q -U "jax[cuda12]"

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)  # constantB_tools.py also sets this on import;
                                            # done here too so this check is meaningful standalone.
import jax.numpy as jnp

print("jax version:", jax.__version__)
print("devices:", jax.devices())
print("x64 check (expect float64):", jnp.array(1.0).dtype)

assert jax.devices()[0].platform == "gpu", (
    "JAX is not seeing a GPU -- re-check the accelerator setting and the pip install above."
)

If this cell has already run once with GPU memory OOM errors before the preallocation fix above
was in place, **restart the session** (Run > Restart Session) before continuing -- the kernel's
own `jax.devices()` call here holds whatever CUDA memory it claimed for the life of the kernel
process, and a stale claim from an earlier attempt will starve the run cell further down.

In [ ]:
import os

REPO_DIR = "/kaggle/working/jax_constantB"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/alfredmallet/jax_constantB.git {REPO_DIR}
else:
    print("repo already present at", REPO_DIR, "-- skipping clone")

## Resuming across sessions

`amplitude_quest.py` is resumable: it checks whether `--state` already exists and continues from
there rather than restarting, and appends to `--csv`. Both are pointed at `/kaggle/working/` below
so they show up as outputs when you **Save Version**.

Kaggle interactive sessions are capped (~9-12h continuous, 30h GPU/week) and `/kaggle/working` does
**not** carry over to a fresh session by itself. To resume a run started in an earlier session:
1. Save Version on the session that has progress you want to keep.
2. In the new session, add that previous version's output as an input dataset.
3. Copy `quest.npz` and `quest.csv` from the added dataset into `/kaggle/working/` before running
   the cell below -- it will then pick up exactly where it left off.

## Dealias (Galerkin 2/3-rule) mode

`DEALIAS` below controls whether the underlying `constantB_tools.py` solver runs
the strict Galerkin 2/3-rule truncation (`Solver(shape, dealias=True)`) or the
original pointwise-collocation formulation. It is passed straight through to
`amplitude_quest.py`'s `--dealias` flag in the run cell below.

**Default is `True` for new runs.** With dealias on, the working-grid `res`
column is the exact (retained-band) Galerkin residual rather than the
pointwise-collocation residual, and each step's honest single-grid
convergence diagnostic is the discarded-band tail of `|B|^2-1`
(`Solver.tail_norm`), not just the spectral-tail fraction already logged.

**Important**: any `quest.npz`/`quest.csv` produced by an *earlier* run of
this notebook (or of the non-jax reference tool) were produced with plain
collocation (`dealias=False`). Resuming those files with `DEALIAS = True`
mixes two different residual/tail semantics in one CSV and silently changes
what `--res-ok`/`--tail-max` are being compared against mid-run. If you have
an existing collocation state you want to keep, either continue with
`DEALIAS = False`, or start a fresh `--state`/`--csv` pair for the dealias
branch.

In [ ]:
# Galerkin (2/3-rule) dealiasing mode -- see markdown above.
# Default True for new runs; set False to reproduce the original
# collocation behaviour (e.g. to keep resuming an old quest.npz/quest.csv).
DEALIAS = True
DEALIAS_FLAG = "--dealias" if DEALIAS else ""
print("DEALIAS:", DEALIAS)

In [ ]:
%cd /kaggle/working/jax_constantB

!XLA_PYTHON_CLIENT_PREALLOCATE=false XLA_PYTHON_CLIENT_ALLOCATOR=platform python3 -u amplitude_quest.py \
    --state /kaggle/working/quest.npz \
    --csv /kaggle/working/quest.csv \
    --grid0 32 32 64 \
    --grid-max 96 96 192 \
    --eps-max 5.0 \
    --de 0.04 \
    --sweeps 6 \
    --cgit 600 \
    --pcg {DEALIAS_FLAG} \
    2>&1 | tee -a /kaggle/working/quest_log.txt

Edit `--grid-max`/`--eps-max` to taste -- e.g. run a short smoke test first
(`--eps-max 0.3 --grid-max 32 32 64`) to confirm the GPU path is faster than CPU on your session
before committing 30-hour-quota GPU time to the full 96x96x192 branch.

The `!python3 ...` cell also sets the preallocation env vars inline (redundant with the `%env`
cell above, but harmless) in case this cell is ever run in a fresh kernel without re-running
cell 2 first.